# 08 - Agente: orquestador multiagente con Claude (local)

Rama: `07MultiAgentIntegration`. Este cuaderno se ejecuta **en la PC** (VS Code con el kernel de `.venv`), no en Colab: usa Claude Code con la membresia del equipo.

| Agente | Quien lo cumple | Que recibe | Que entrega |
|---|---|---|---|
| Planificador | Claude, aislado y sin herramientas | la pregunta y la ruta del archivo | plan JSON (validado por Python) |
| Ejecutor | Python (`ptcg_agente/herramientas/`) | el plan | resultados del modelo final |
| Verificador | Python, deterministico | cada resultado y el texto final | aprobado o lista de problemas |
| Redactor | Claude, aislado y sin herramientas | solo resultados aprobados | respuesta para el usuario |

Requisitos (`docx/05AmbienteDeDesarrollo/GuiaDelAgente.md`): `.venv` con `requirements.txt`, artefactos bajados con `python -m ptcg_agente.artefactos` y Claude Code instalado con la sesion iniciada.

In [ ]:
# PREPARACION (local)
# PROPOSITO: ubicar la raiz del repositorio, cargar el paquete del agente y comprobar que
# Claude Code esta instalado.
# ESPERADO: la ruta del repo, la rama, el commit y la version de Claude Code.
import json
import shutil
import subprocess
import sys
from datetime import datetime
from pathlib import Path

REPO_DIR = Path.cwd()
while not (REPO_DIR / ".mcp.json").exists():
    if REPO_DIR.parent == REPO_DIR:
        raise RuntimeError("Abre el cuaderno desde la carpeta del repositorio")
    REPO_DIR = REPO_DIR.parent
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
for nombre in [m for m in list(sys.modules) if m.split(".")[0] in ("ptcg_wp", "ptcg_agente")]:
    del sys.modules[nombre]

CLAUDE = shutil.which("claude")
assert CLAUDE, "No se encontro 'claude': instala Claude Code (GuiaDelAgente, seccion 3.1) y reinicia VS Code"
version = subprocess.run([CLAUDE, "--version"], capture_output=True, text=True).stdout.strip()
git = lambda *a: subprocess.check_output(["git", "-C", str(REPO_DIR), *a], text=True).strip()
print("repo:", REPO_DIR)
print("rama:", git("branch", "--show-current"), "| commit:", git("rev-parse", "--short", "HEAD"))
print("Claude Code:", version)

In [ ]:
# PRUEBAS DEL AGENTE
# PROPOSITO: correr las pruebas automaticas del agente (herramientas, verificador, artefactos,
# registro, servidor MCP y orquestador con un LLM de prueba) y del modelo exportado.
# ESPERADO: "N passed" sin "failed" y la linea "OK: pruebas del agente aprobadas".
resultado = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_agente.py",
                            "tests/test_predictor.py"], cwd=REPO_DIR, capture_output=True, text=True)
print(resultado.stdout[-2000:], resultado.stderr[-2000:])
assert resultado.returncode == 0, "fallaron pruebas del agente: revisa la salida"
print("OK: pruebas del agente aprobadas")

In [ ]:
# MODELO Y EJEMPLOS
# PROPOSITO: cargar el modelo final verificando SHA-256 y version de scikit-learn, y elegir una
# repeticion real de ejemplo con estados de las tres etapas.
# ESPERADO: el SHA-256 del modelo, la partida elegida y los pasos que se consultan.
from ptcg_agente import orquestador
from ptcg_agente.artefactos import ARTEFACTOS_DIR
from ptcg_agente.entradas import leer_json
from ptcg_agente.modelo import ModeloAgente
from ptcg_agente.registro import Registro
from ptcg_wp.features import acting_player

MODELO = ModeloAgente.cargar()
REGISTRO = Registro("orquestador", MODELO.modelo_sha256, MODELO.commit)
print("modelo:", MODELO.modelo_sha256[:16], "| sesion:", REGISTRO.sesion)

EJEMPLOS = sorted((ARTEFACTOS_DIR / "agente" / "ejemplos").glob("partida_*.json"))
assert EJEMPLOS, "No hay ejemplos: ejecuta python -m ptcg_agente.artefactos (GuiaDelAgente, seccion 2)"
ARCHIVO = str(EJEMPLOS[0])
partida = leer_json(ARCHIVO)


def turno(paso):
    j = acting_player(paso)
    cur = (paso[j]["observation"].get("current") or {}) if j is not None else {}
    return cur.get("turn")


turnos = [turno(p) for p in partida["steps"]]
PASO_FINAL = max(i for i, t in enumerate(turnos) if t is not None and t >= 10)
PASO_INICIO = min(i for i, t in enumerate(turnos) if t is not None and 1 <= t <= 5)
print("partida:", Path(ARCHIVO).name, "| pasos:", len(turnos))
print("paso de etapa final:", PASO_FINAL, "(turno", turnos[PASO_FINAL], ") | paso de inicio:",
      PASO_INICIO, "(turno", turnos[PASO_INICIO], ")")


def mostrar(r):
    print(f"PLAN ({r['fuente_plan']}):")
    for i, s in enumerate(r["plan"]["pasos"]):
        args = {k: (Path(v).name if k == "archivo" else v) for k, v in s["argumentos"].items()}
        print(f"  {i + 1}. {s['herramienta']} {args} - {s['motivo']}")
    for ronda in r["rondas_redactor"]:
        v = ronda["verificacion"]
        print(f"redactor, ronda {ronda['ronda']}: {'aprobada' if v['aprobado'] else 'rechazada'}"
              f" ({v['cifras_revisadas']} cifras)" + (f" -> {v['problemas']}" if v["problemas"] else ""))
    print(f"\nRESPUESTA ({r['fuente_respuesta']}):\n")
    print(r["respuesta"])

In [ ]:
# CONSULTA 1: ESTADO DE ETAPA FINAL
# PROPOSITO: preguntar quien va ganando en un estado concreto y por que.
# ESPERADO: un plan con estimar_probabilidad y explicar_prediccion, una respuesta del redactor
# aprobada por el verificador y el por que en premios, descartes, mazo o tablero.
r1 = orquestador.consultar("¿Quien va ganando en este momento y por que?", ARCHIVO, PASO_FINAL,
                           modelo=MODELO, registro=REGISTRO)
mostrar(r1)

In [ ]:
# CONSULTA 2: PARTIDA COMPLETA
# PROPOSITO: pedir la evolucion de toda la partida y sus momentos decisivos.
# ESPERADO: un plan con evaluar_partida y una respuesta con la probabilidad por turno y los
# puntos de giro, aprobada por el verificador.
r2 = orquestador.consultar("¿Como evoluciono la partida y cuales fueron los momentos decisivos?",
                           ARCHIVO, modelo=MODELO, registro=REGISTRO)
mostrar(r2)

In [ ]:
# CONSULTA 3: CONFIABILIDAD AL INICIO
# PROPOSITO: comprobar que el agente dice los limites cuando el modelo es menos confiable.
# ESPERADO: un plan que incluye describir_modelo o revisar_dominio y una respuesta que menciona
# la advertencia de etapa de inicio.
r3 = orquestador.consultar("¿Que tan confiable es la prediccion en este momento de la partida?",
                           ARCHIVO, PASO_INICIO, modelo=MODELO, registro=REGISTRO)
mostrar(r3)

In [ ]:
# EVIDENCIA DEL ORQUESTADOR
# PROPOSITO: copiar las lineas de esta sesion del registro a docx/03Evidencia/ y resumir que
# agente hizo que.
# ESPERADO: la tabla de invocaciones por agente y etapa, y la ruta del archivo de evidencia.
import pandas as pd

lineas = [json.loads(x) for x in REGISTRO.archivo.read_text(encoding="utf-8").splitlines()]
sesion = [x for x in lineas if x["sesion"] == REGISTRO.sesion]
destino = REPO_DIR / "docx" / "03Evidencia" / f"{datetime.now():%d-%m-%y}_RegistroOrquestador.jsonl"
destino.write_text("".join(json.dumps(x, ensure_ascii=False, default=str) + "\n" for x in sesion),
                   encoding="utf-8")

tabla = pd.DataFrame([{"agente": x["agente"], "etapa": x["herramienta"], "ms": x["duracion_ms"]}
                      for x in sesion])
print(tabla.groupby(["agente", "etapa"]).agg(llamadas=("ms", "size"), ms_medio=("ms", "mean")).round(1))
costo = sum((x["salida"].get("meta") or {}).get("total_cost_usd") or 0 for x in sesion
            if x["agente"] in ("planificador", "redactor"))
print(f"\nrespuestas aprobadas: {sum(r['verificacion']['aprobado'] for r in (r1, r2, r3))} de 3")
print(f"costo equivalente informado por Claude Code: {costo:.4f} USD (cubierto por la membresia)")
print("evidencia:", destino.relative_to(REPO_DIR))

## Lo mismo desde la terminal o desde Claude Code

- **Terminal** (raiz del repo, `.venv` activo): `PYTHONPATH=src python -m ptcg_agente.orquestador "¿Quien va ganando?" --archivo artefactos_agente/agente/ejemplos/partida_XXXX.json --paso 120` (en PowerShell: `$env:PYTHONPATH="src"` antes del comando).
- **Claude Code con subagentes:** abrir `claude` en la raiz del repo y escribir `/consulta-ptcg artefactos_agente/agente/ejemplos/partida_XXXX.json 120 ¿Quien va ganando?`. Los subagentes estan en `.claude/agents/`.